# 1. RAG Básico - Implementación con Streamlit

## Objetivos de Aprendizaje
- Comprender los componentes fundamentales de un sistema RAG
- Implementar una búsqueda por palabras clave simple
- Generar respuestas basadas en contexto recuperado
- Construir una interfaz interactiva con Streamlit

## ¿Qué es RAG (Retrieval-Augmented Generation)?

RAG es una arquitectura que combina dos fases para generar respuestas:
1. **Retrieval (Recuperación)**: Busca y recupera información relevante de una base de conocimiento.
2. **Generation (Generación)**: Utiliza la información recuperada como contexto para que un LLM genere una respuesta precisa y fundamentada.

### Ventajas:
- **Reduce alucinaciones**: Las respuestas se basan en datos concretos.
- **Conocimiento actualizado**: La base de conocimiento puede actualizarse fácilmente.
- **Transparencia**: Se puede citar la fuente de la información.

### Componentes de nuestro RAG Básico:
- **Base de documentos**: Una lista de strings en memoria.
- **Mecanismo de retrieval**: Búsqueda simple por coincidencia de palabras clave.
- **Generador**: Un LLM (GPT-4o) que recibe el contexto y la pregunta.

# Instalación de dependencias
!pip install openai streamlit

In [1]:
!pip install openai streamlit


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\ingri\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [2]:
# Celda 2: Importaciones y configuración
import os
from openai import OpenAI
import json

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


In [3]:
# Inicialización del cliente OpenAI (Groq)
def initialize_client():
    client = OpenAI(
        base_url=os.getenv("GROQ_BASE_URL"),
        api_key=os.getenv("GROQ_API_KEY")
    )
    return client

client = initialize_client()
print("✅ Cliente OpenAI inicializado")

✅ Cliente OpenAI inicializado


# Base de documentos

Cargamos la base de documentos que vamos a utilizar para el RAG. En este caso, vamos a utilizar una base de documentos que contiene información sobre la historia de Argentina.

In [4]:
documents = [
    "La inteligencia artificial es una rama de la informática que busca crear máquinas capaces de realizar tareas que requieren inteligencia humana.",
    "Los modelos de lenguaje grande (LLM) son sistemas de IA entrenados en enormes cantidades de texto para generar y comprender lenguaje natural.",
    "RAG (Retrieval-Augmented Generation) combina la búsqueda de información relevante con la generación de texto para producir respuestas más precisas.",
    "LangChain es un framework que facilita el desarrollo de aplicaciones con modelos de lenguaje, proporcionando herramientas para cadenas y agentes.",
    "El prompt engineering es la práctica de diseñar instrucciones efectivas para obtener los mejores resultados de los modelos de IA.",
    "la capital de Francia es paris"
]

print(f"📚 Base de documentos creada con {len(documents)} documentos:")
for i, doc in enumerate(documents, 1):
    print(f"{i}. {doc[:60]}...")

📚 Base de documentos creada con 6 documentos:
1. La inteligencia artificial es una rama de la informática que...
2. Los modelos de lenguaje grande (LLM) son sistemas de IA entr...
3. RAG (Retrieval-Augmented Generation) combina la búsqueda de ...
4. LangChain es un framework que facilita el desarrollo de apli...
5. El prompt engineering es la práctica de diseñar instruccione...
6. la capital de Francia es paris...


---
## Desglose paso a paso de la lógica interna

Antes de seguir, vamos a probar por separado cada pieza de la línea `if any(word in doc.lower() for word in query_lower.split())`, para entender cómo se arma esa condición.


### Paso 1: ¿Qué hace el operador `in`?

El operador `in` revisa si un texto está contenido dentro de otro, letra por letra (no palabra por palabra). Siempre devuelve `True` o `False`, por sí solo.


In [ ]:
"capital" in "la capital de francia"  # → True, solo, sin nada más alrededor
#"perro" in "la capital de francia"    # → False


### Paso 2: ¿Qué hace `.split()`?

Divide un texto en una lista de palabras separadas, cortando por los espacios. Es lo que nos permite comparar palabra por palabra en vez de la frase completa.


In [ ]:
print("Capital".lower().split())
print("qué es la inteligencia artificial".lower().split())


### Paso 3: Uniendo todo con `any()`

`any()` revisa, una por una, si alguna palabra de la pregunta (`query_lower.split()`) está contenida en el documento. Apenas encuentra una sola coincidencia, devuelve `True` y se detiene — no necesita que coincidan todas las palabras.


In [ ]:
print(any(word in doc.lower() for word in query_lower.split()))

True


: 

---

# Función de recuperación simple

La función simple_retrieval() es una función que toma una pregunta y un índice, y devuelve una lista de documentos relevantes. En este caso, la función simplemente devuelve los primeros 3 documentos del índice.

In [30]:
def simple_retrieval(query, documents):
    relevant_docs = []
    query_lower = query.lower()
    
    for doc in documents: 
        if any(word in doc.lower() for word in query_lower.split()): # recorre cada palabra de la pregunta 
            relevant_docs.append(doc)
    
    return relevant_docs[:3]

# Prueba de la función de recuperación
test_query = "Capital"
relevant_docs = simple_retrieval(test_query, documents)

print(f"🔍 Prueba de recuperación para: '{test_query}'")
print(f"Documentos relevantes encontrados: {len(relevant_docs)}")
for i, doc in enumerate(relevant_docs, 1):
    print(f"{i}. {doc}")

🔍 Prueba de recuperación para: 'Capital'
Documentos relevantes encontrados: 1
1. la capital de Francia es paris


# Función de generación de respuesta
La función generate_response se encarga de generar una respuesta a partir de una pregunta y un contexto. Para ello, utiliza un modelo de lenguaje preentrenado y una plantilla de prompt.

In [28]:
def generate_response(client, query, context):
    prompt = f"""Contexto:
{context}

Pregunta: {query}

Responde basándote únicamente en el contexto proporcionado. Si la información no está en el contexto, indica que no tienes esa información."""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=500
    )
    
    return response.choices[0].message.content

print("✅ Función de generación definida")

✅ Función de generación definida


# Sistema RAG completo - Ejemplo 1

In [ ]:
query1 = "¿Qué es la inteligencia artificial?"

print(f"❓ Pregunta: {query1}")
print("\n" + "="*50)

# Paso 1: Recuperación
relevant_docs = simple_retrieval(query1, documents)
print(f"📋 Documentos relevantes ({len(relevant_docs)}):")
for i, doc in enumerate(relevant_docs, 1):
    print(f"{i}. {doc}")

# Paso 2: Generación
if relevant_docs:  # si la lista no esta vacia 
    context = "\n".join(relevant_docs) # crear un contexto con los documentos relevantes
    response = generate_response(client, query1, context)
    print(f"\n🤖 Respuesta:")
    print(response)
else:
    print("❌ No se encontraron documentos relevantes")

❓ Pregunta: ¿Qué es la inteligencia artificial?

📋 Documentos relevantes (3):
1. La inteligencia artificial es una rama de la informática que busca crear máquinas capaces de realizar tareas que requieren inteligencia humana.
2. Los modelos de lenguaje grande (LLM) son sistemas de IA entrenados en enormes cantidades de texto para generar y comprender lenguaje natural.
3. RAG (Retrieval-Augmented Generation) combina la búsqueda de información relevante con la generación de texto para producir respuestas más precisas.



🤖 Respuesta:
La inteligencia artificial es una rama de la informática que busca crear máquinas capaces de realizar tareas que requieren inteligencia humana.


# Sistema RAG completo - Ejemplo 2

In [7]:
query2 = "¿ Cómo funciona RAG ?"

print(f"❓ Pregunta: {query2}")
print("\n" + "="*50)

# Paso 1: Recuperación
relevant_docs = simple_retrieval(query2, documents)
print(f"📋 Documentos relevantes ({len(relevant_docs)}):")
for i, doc in enumerate(relevant_docs, 1):
    print(f"{i}. {doc}")

# Paso 2: Generación
if relevant_docs:
    context = "\n".join(relevant_docs)
    response = generate_response(client, query2, context)
    print(f"\n🤖 Respuesta:")
    print(response)
else:
    print("❌ No se encontraron documentos relevantes")

❓ Pregunta: ¿ Cómo funciona RAG ?

📋 Documentos relevantes (1):
1. RAG (Retrieval-Augmented Generation) combina la búsqueda de información relevante con la generación de texto para producir respuestas más precisas.



🤖 Respuesta:
Según el contexto proporcionado, RAG (Retrieval-Augmented Generation) funciona combinando la búsqueda de información relevante con la generación de texto para producir respuestas más precisas. No tengo más información sobre su funcionamiento específico en el contexto dado.


# Sistema RAG completo - Ejemplo 3 (pregunta sin contexto)

In [12]:
query3 = "capital"

print(f"❓ Pregunta: {query3}")
print("\n" + "="*50)

# Paso 1: Recuperación
relevant_docs = simple_retrieval(query3, documents)
print(f"📋 Documentos relevantes ({len(relevant_docs)}):")
for i, doc in enumerate(relevant_docs, 1):
    print(f"{i}. {doc}")

# Paso 2: Generación
if relevant_docs:
    context = "\n".join(relevant_docs)
    response = generate_response(client, query3, context)
    print(f"\n🤖 Respuesta:")
    print(response)
else:
    print("❌ No se encontraron documentos relevantes")
    print("🤖 Respuesta: No tengo información sobre esa pregunta en mi base de documentos.")

❓ Pregunta: capital

📋 Documentos relevantes (1):
1. la capital de Francia es paris

🤖 Respuesta:
Según el contexto proporcionado, la capital es París.


In [ ]:
"capital" in "la capital de francia"  # → True, solo, sin nada más alrededor
"perro" in "la capital de francia"    # → False
